# 03. Modeling
Baseline, градиентный бустинг, тюнинг, MLflow.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


## 1. Загрузка processed данных и split


In [2]:
import pandas as pd
import mlflow

from src.features import get_train_test_split
from src.train import (
    train_baseline, train_catboost, train_xgboost, train_lightgbm,
    tune_with_optuna, save_best_model, MLFLOW_EXPERIMENT,
)

df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'telco_processed.csv')
X_train, X_test, y_train, y_test = get_train_test_split(df)
mlflow.set_experiment(MLFLOW_EXPERIMENT)


<Experiment: artifact_location='file:///E:/.PetProjects/Churn%20Predictions/notebooks/mlruns/137816881767761797', creation_time=1782240202461, experiment_id='137816881767761797', last_update_time=1782240202461, lifecycle_stage='active', name='telco-churn', tags={}>

## 2. Baseline: Logistic Regression


In [3]:
lr_model, lr_metrics = train_baseline(X_train, y_train, X_test, y_test)
lr_metrics


E:\.PetProjects\Churn Predictions\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


2026/06/23 21:59:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


{'accuracy': 0.7423704755145494,
 'precision': np.float64(0.5093378607809848),
 'recall': np.float64(0.8021390374331551),
 'f1': np.float64(0.6230529595015576),
 'roc_auc': np.float64(0.8457206334444186),
 'pr_auc': np.float64(0.6566975135302235)}

## 3. Градиентный бустинг: CatBoost / XGBoost / LightGBM


In [4]:
cb_model, cb_metrics = train_catboost(X_train, y_train, X_test, y_test)
xgb_model, xgb_metrics = train_xgboost(X_train, y_train, X_test, y_test)
lgbm_model, lgbm_metrics = train_lightgbm(X_train, y_train, X_test, y_test)
cb_metrics, xgb_metrics, lgbm_metrics


2026/06/23 21:59:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


E:\.PetProjects\Churn Predictions\venv\Lib\site-packages\xgboost\core.py:158: UserWarning: [21:59:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


2026/06/23 21:59:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000188 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/06/23 21:59:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


({'accuracy': 0.7608232789212207,
  'precision': np.float64(0.5357833655705996),
  'recall': np.float64(0.7406417112299465),
  'f1': np.float64(0.621773288439955),
  'roc_auc': np.float64(0.8397207367795603),
  'pr_auc': np.float64(0.6483919617233384)},
 {'accuracy': 0.7615330021291696,
  'precision': np.float64(0.5351851851851852),
  'recall': np.float64(0.7727272727272727),
  'f1': np.float64(0.6323851203501094),
  'roc_auc': np.float64(0.8388901805781601),
  'pr_auc': np.float64(0.6499694960919582)},
 {'accuracy': 0.758694109297374,
  'precision': np.float64(0.5338645418326693),
  'recall': np.float64(0.7165775401069518),
  'f1': np.float64(0.6118721461187214),
  'roc_auc': np.float64(0.8305445761967502),
  'pr_auc': np.float64(0.6329581214127019)})

## 4. Сравнение моделей
Таблица метрик + сравнение class_weight vs SMOTE (TODO).


In [5]:
comparison = pd.DataFrame({
    'logistic_regression': lr_metrics,
    'catboost': cb_metrics,
    'xgboost': xgb_metrics,
    'lightgbm': lgbm_metrics,
}).T
comparison.sort_values('pr_auc', ascending=False)


,accuracy,precision,recall,f1,roc_auc,pr_auc
logistic_regression,0.742370,0.509338,0.802139,0.623053,0.845721,0.656698
xgboost,0.761533,0.535185,0.772727,0.632385,0.838890,0.649969
catboost,0.760823,0.535783,0.740642,0.621773,0.839721,0.648392
lightgbm,0.758694,0.533865,0.716578,0.611872,0.830545,0.632958


## 5. Тюнинг гиперпараметров (Optuna)


In [6]:
best_params, best_pr_auc = tune_with_optuna(X_train, y_train, X_test, y_test, n_trials=30)
best_params, best_pr_auc


[I 2026-06-23 21:59:29,877] A new study created in memory with name: no-name-62344a0e-b767-40de-bed6-961050caa0ee


[I 2026-06-23 21:59:30,125] Trial 0 finished with value: 0.6206521199374455 and parameters: {'n_estimators': 552, 'max_depth': 8, 'learning_rate': 0.05541847430287673, 'num_leaves': 25}. Best is trial 0 with value: 0.6206521199374455.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000161 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000155 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2026-06-23 21:59:30,394] Trial 1 finished with value: 0.6157052720161977 and parameters: {'n_estimators': 226, 'max_depth': 10, 'learning_rate': 0.06859014794345465, 'num_leaves': 125}. Best is trial 0 with value: 0.6206521199374455.


...

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000153 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2026-06-23 21:59:36,531] Trial 28 finished with value: 0.6562558568913553 and parameters: {'n_estimators': 346, 'max_depth': 7, 'learning_rate': 0.010087516570022608, 'num_leaves': 46}. Best is trial 23 with value: 0.6605681833848472.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-06-23 21:59:36,691] Trial 29 finished with value: 0.6617405238589581 and parameters: {'n_estimators': 518, 'max_depth': 3, 'learning_rate': 0.02356162860707011, 'num_leaves': 90}. Best is trial 29 with value: 0.6617405238589581.


({'n_estimators': 518,
  'max_depth': 3,
  'learning_rate': 0.02356162860707011,
  'num_leaves': 90},
 0.6617405238589581)

## 6. Сравнение запусков в MLflow


In [7]:
runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT])
runs.sort_values('metrics.pr_auc', ascending=False)[['tags.mlflow.runName', 'metrics.pr_auc', 'metrics.roc_auc', 'metrics.recall']].head(10)


,tags.mlflow.runName,metrics.pr_auc,metrics.roc_auc,metrics.recall
52,optuna_trial_12,0.665621,NaN,NaN
43,optuna_trial_21,0.665612,NaN,NaN
42,optuna_trial_22,0.665408,NaN,NaN
53,optuna_trial_11,0.664832,NaN,NaN
51,optuna_trial_13,0.663733,NaN,NaN
0,optuna_trial_29,0.661741,NaN,NaN
6,optuna_trial_23,0.660568,NaN,NaN
11,optuna_trial_18,0.660535,NaN,NaN
8,optuna_trial_21,0.660421,NaN,NaN
41,optuna_trial_23,0.660194,NaN,NaN


## 7. Сохранение лучшей модели


In [8]:
from lightgbm import LGBMClassifier
from src.train import compute_metrics, save_best_model

best_lgbm_tuned = LGBMClassifier(**best_params, class_weight="balanced", random_state=42)
best_lgbm_tuned.fit(X_train, y_train)
lgbm_tuned_metrics = compute_metrics(
    y_test,
    best_lgbm_tuned.predict(X_test),
    best_lgbm_tuned.predict_proba(X_test)[:, 1],
)

all_models = {
    "logistic_regression": (lr_model, lr_metrics),
    "catboost": (cb_model, cb_metrics),
    "xgboost": (xgb_model, xgb_metrics),
    "lightgbm": (lgbm_model, lgbm_metrics),
    "lightgbm_tuned": (best_lgbm_tuned, lgbm_tuned_metrics),
}
best_name = max(all_models, key=lambda n: all_models[n][1]["pr_auc"])
best_obj, best_metrics_final = all_models[best_name]
print(f"Best model: {best_name}")
print(f"PR-AUC: {best_metrics_final['pr_auc']:.4f}")
save_best_model(best_obj, best_name, best_metrics_final)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 910
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

## 8. Результаты
TODO: перенести итоговую таблицу метрик в README.
